In [1]:
import numpy as np

import transpose_invariance as tpi

import skimage as ski
import skimage.registration as skr

See the analysis in `ai_output/gemini_flow_invariance.md`.

In [2]:
imgs = list(tpi.get_3d_images())
for i in (0, 1):
    imgs[0] = ski.img_as_float(imgs[0][:10, :64, :64])
img = imgs[0]

In [3]:
img.shape

(10, 64, 64)

In [4]:
def f_tvl1(fixed):
    moving = np.roll(fixed, shift=(1, 2, 3), axis=(0, 1, 2))
    return skr.optical_flow_tvl1(fixed, moving)

def f_ilk(fixed):
    moving = np.roll(fixed, shift=(1, 2, 3), axis=(0, 1, 2))
    return skr.optical_flow_ilk(fixed, moving)

In [5]:
ws_orig = f_tvl1(img)

In [6]:
def check_func(img1, img2):
    assert np.allclose(img1, img2)

In [10]:
def rolled_proc_flow(img, axes, func):
    r_img = np.transpose(img, axes)
    f_r_img = func(r_img)
    # Roll and resort axes, coords
    c_last = np.moveaxis(f_r_img, 0, -1)
    back_axes = list(np.argsort(axes))
    clt = np.transpose(c_last, back_axes + [-1])
    c_first = np.moveaxis(clt, -1, 0)
    # Reorder coordinates.
    return c_first[back_axes]

In [12]:
ws_rolled = rolled_proc_flow(img, (2, 1, 0), f_tvl1)
np.max(np.abs(ws_orig - ws_rolled))
# check_func(ws_orig, ws_rolled)

np.float32(0.18567322)

In [15]:
ws_orig[:, 0, 0, 0], ws_rolled[:, 0, 0, 0]

(array([0.02646505, 0.0240528 , 0.14187619], dtype=float32),
 array([-0.00355637,  0.04161558,  0.192557  ], dtype=float32))

In [ ]:
# All images are transpose invariant
tpi.assert_all_orders(imgs, f_tvl1, chk_func=check_func)